In [19]:
!pip -q install requests beautifulsoup4 lxml pandas tqdm

Imports & config

In [20]:
import os
import re
import time
import json
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Project paths
PROJECT_DIR = "/content/drive/MyDrive/finance-rag-analyst"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/finance-rag-analyst


SEC headers

In [21]:
SEC_USER_AGENT = "Anish Kale akale014@ucr.edu"

HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

DATA_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

ARCHIVE_HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

REQUEST_SLEEP = 0.15

Helper request function

In [22]:
def safe_get(url, headers=None, sleep=REQUEST_SLEEP, max_retries=3):
    """
    SEC-friendly GET request with basic retry and throttling.
    """
    for attempt in range(max_retries):
        try:
            time.sleep(sleep)
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                return response

            if response.status_code in [429, 403, 503]:
                wait = 2 ** attempt
                print(f"Rate/Access issue {response.status_code}. Waiting {wait}s. URL: {url}")
                time.sleep(wait)
                continue

            print(f"Request failed: {response.status_code} | {url}")
            return response

        except Exception as e:
            wait = 2 ** attempt
            print(f"Error: {e}. Waiting {wait}s.")
            time.sleep(wait)

    raise RuntimeError(f"Failed after retries: {url}")

Get ticker-to-CIK map

In [23]:
def load_company_tickers():
    url = "https://www.sec.gov/files/company_tickers.json"
    response = safe_get(url, headers=HEADERS)
    data = response.json()

    records = []
    for _, item in data.items():
        records.append({
            "ticker": item["ticker"].upper(),
            "company": item["title"],
            "cik": str(item["cik_str"]).zfill(10)
        })

    return pd.DataFrame(records)

ticker_df = load_company_tickers()
ticker_df.head()

,ticker,company,cik
0,NVDA,NVIDIA CORP,0001045810
1,AAPL,Apple Inc.,0000320193
2,GOOGL,Alphabet Inc.,0001652044
3,MSFT,MICROSOFT CORP,0000789019
4,AMZN,AMAZON COM INC,0001018724


Choose MVP tickers

In [24]:
TICKERS = ["NVDA", "MSFT", "AAPL", "AMZN", "GOOGL"]

universe = ticker_df[ticker_df["ticker"].isin(TICKERS)].copy()
universe

,ticker,company,cik
0,NVDA,NVIDIA CORP,0001045810
1,AAPL,Apple Inc.,0000320193
2,GOOGL,Alphabet Inc.,0001652044
3,MSFT,MICROSOFT CORP,0000789019
4,AMZN,AMAZON COM INC,0001018724


Pull recent SEC filing metadata

In [25]:
def get_company_submissions(cik):
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = safe_get(url, headers=DATA_HEADERS)
    return response.json()


def extract_recent_filings(submission_json, ticker, company, forms=("10-K", "10-Q"), max_filings_per_form=3):
    recent = submission_json["filings"]["recent"]

    df = pd.DataFrame({
        "accession_number": recent["accessionNumber"],
        "filing_date": recent["filingDate"],
        "report_date": recent["reportDate"],
        "form_type": recent["form"],
        "primary_document": recent["primaryDocument"],
    })

    df = df[df["form_type"].isin(forms)].copy()
    df["ticker"] = ticker
    df["company"] = company
    df["cik"] = submission_json["cik"]
    df["cik_padded"] = str(submission_json["cik"]).zfill(10)

    selected = []
    for form in forms:
        selected.append(df[df["form_type"] == form].head(max_filings_per_form))

    return pd.concat(selected, ignore_index=True)


all_filings = []

for _, row in tqdm(universe.iterrows(), total=len(universe)):
    sub = get_company_submissions(row["cik"])
    filings = extract_recent_filings(
        sub,
        ticker=row["ticker"],
        company=row["company"],
        forms=("10-K", "10-Q"),
        max_filings_per_form=3
    )
    all_filings.append(filings)

filings_df = pd.concat(all_filings, ignore_index=True)
filings_df

  0%|          | 0/5 [00:00<?, ?it/s]

,accession_number,filing_date,report_date,form_type,primary_document,ticker,company,cik,cik_padded
0,0001045810-26-000021,2026-02-25,2026-01-25,10-K,nvda-20260125.htm,NVDA,NVIDIA CORP,0001045810,0001045810
1,0001045810-25-000023,2025-02-26,2025-01-26,10-K,nvda-20250126.htm,NVDA,NVIDIA CORP,0001045810,0001045810
2,0001045810-24-000029,2024-02-21,2024-01-28,10-K,nvda-20240128.htm,NVDA,NVIDIA CORP,0001045810,0001045810
3,0001045810-26-000052,2026-05-20,2026-04-26,10-Q,nvda-20260426.htm,NVDA,NVIDIA CORP,0001045810,0001045810
4,0001045810-25-000230,2025-11-19,2025-10-26,10-Q,nvda-20251026.htm,NVDA,NVIDIA CORP,0001045810,0001045810
5,0001045810-25-000209,2025-08-27,2025-07-27,10-Q,nvda-20250727.htm,NVDA,NVIDIA CORP,0001045810,0001045810
6,0000320193-25-000079,2025-10-31,2025-09-27,10-K,aapl-20250927.htm,AAPL,Apple Inc.,0000320193,0000320193
7,0000320193-24-000123,2024-11-01,2024-09-28,10-K,aapl-20240928.htm,AAPL,Apple Inc.,0000320193,0000320193
8,0000320193-23-000106,2023-11-03,2023-09-30,10-K,aapl-20230930.htm,AAPL,Apple Inc.,0000320193,0000320193
9,0000320193-26-000013,2026-05-01,2026-03-28,10-Q,aapl-20260328.htm,AAPL,Apple Inc.,0000320193,0000320193


Build SEC filing URLs

In [26]:
def build_filing_url(cik_padded, accession_number, primary_document):
    cik_no_zeros = str(int(cik_padded))
    accession_clean = accession_number.replace("-", "")
    return f"https://www.sec.gov/Archives/edgar/data/{cik_no_zeros}/{accession_clean}/{primary_document}"


filings_df["filing_url"] = filings_df.apply(
    lambda r: build_filing_url(
        r["cik_padded"],
        r["accession_number"],
        r["primary_document"]
    ),
    axis=1
)

filings_df[["ticker", "form_type", "filing_date", "filing_url"]].head()

,ticker,form_type,filing_date,filing_url
0,NVDA,10-K,2026-02-25,https://www.sec.gov/Archives/edgar/data/104581...
1,NVDA,10-K,2025-02-26,https://www.sec.gov/Archives/edgar/data/104581...
2,NVDA,10-K,2024-02-21,https://www.sec.gov/Archives/edgar/data/104581...
3,NVDA,10-Q,2026-05-20,https://www.sec.gov/Archives/edgar/data/104581...
4,NVDA,10-Q,2025-11-19,https://www.sec.gov/Archives/edgar/data/104581...


Dwnld filing HTML/text

In [27]:
def download_filing_text(url):
    response = safe_get(url, headers=ARCHIVE_HEADERS)
    html = response.text

    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "ix:header"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


raw_records = []

for _, row in tqdm(filings_df.iterrows(), total=len(filings_df)):
    try:
        text = download_filing_text(row["filing_url"])

        raw_records.append({
            "ticker": row["ticker"],
            "company": row["company"],
            "cik": row["cik_padded"],
            "form_type": row["form_type"],
            "filing_date": row["filing_date"],
            "report_date": row["report_date"],
            "accession_number": row["accession_number"],
            "primary_document": row["primary_document"],
            "source_url": row["filing_url"],
            "text": text
        })

        print(row["ticker"], row["form_type"], row["filing_date"], "chars:", len(text))

    except Exception as e:
        print("FAILED:", row["ticker"], row["form_type"], row["filing_date"], e)

raw_df = pd.DataFrame(raw_records)
raw_df.head()

  0%|          | 0/30 [00:00<?, ?it/s]

/tmp/ipykernel_3301/4246696777.py:5: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


NVDA 10-K 2026-02-25 chars: 340269
NVDA 10-K 2025-02-26 chars: 348341
NVDA 10-K 2024-02-21 chars: 339967
NVDA 10-Q 2026-05-20 chars: 136174
NVDA 10-Q 2025-11-19 chars: 151162
NVDA 10-Q 2025-08-27 chars: 133221
AAPL 10-K 2025-10-31 chars: 206727
AAPL 10-K 2024-11-01 chars: 204321
AAPL 10-K 2023-11-03 chars: 200598
AAPL 10-Q 2026-05-01 chars: 84747
AAPL 10-Q 2026-01-30 chars: 56972
AAPL 10-Q 2025-08-01 chars: 59899
GOOGL 10-K 2026-02-05 chars: 344469
GOOGL 10-K 2025-02-05 chars: 350760
GOOGL 10-K 2024-01-31 chars: 340571
GOOGL 10-Q 2026-04-30 chars: 151898
GOOGL 10-Q 2025-10-30 chars: 155883
GOOGL 10-Q 2025-07-24 chars: 153108
MSFT 10-K 2025-07-30 chars: 313398
MSFT 10-K 2024-07-30 chars: 354483
MSFT 10-K 2023-07-27 chars: 337257
MSFT 10-Q 2026-04-29 chars: 205633
MSFT 10-Q 2026-01-28 chars: 201982
MSFT 10-Q 2025-10-29 chars: 185360
AMZN 10-K 2026-02-06 chars: 283477
AMZN 10-K 2025-02-07 chars: 280184
AMZN 10-K 2024-02-02 chars: 283111
AMZN 10-Q 2026-04-30 chars: 178302
AMZN 10-Q 2025-10

,ticker,company,cik,form_type,filing_date,report_date,accession_number,primary_document,source_url,text
0,NVDA,NVIDIA CORP,0001045810,10-K,2026-02-25,2026-01-25,0001045810-26-000021,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/104581...,nvda-20260125 Table of Contents UNITED STATES ...
1,NVDA,NVIDIA CORP,0001045810,10-K,2025-02-26,2025-01-26,0001045810-25-000023,nvda-20250126.htm,https://www.sec.gov/Archives/edgar/data/104581...,nvda-20250126 Table of Contents UNITED STATES ...
2,NVDA,NVIDIA CORP,0001045810,10-K,2024-02-21,2024-01-28,0001045810-24-000029,nvda-20240128.htm,https://www.sec.gov/Archives/edgar/data/104581...,nvda-20240128 Table of Contents UNITED STATES ...
3,NVDA,NVIDIA CORP,0001045810,10-Q,2026-05-20,2026-04-26,0001045810-26-000052,nvda-20260426.htm,https://www.sec.gov/Archives/edgar/data/104581...,nvda-20260426 UNITED STATES SECURITIES AND EXC...
4,NVDA,NVIDIA CORP,0001045810,10-Q,2025-11-19,2025-10-26,0001045810-25-000230,nvda-20251026.htm,https://www.sec.gov/Archives/edgar/data/104581...,nvda-20251026 UNITED STATES SECURITIES AND EXC...


Save raw filings

In [28]:
raw_path = f"{RAW_DIR}/sec_filings_raw.csv"
raw_df.to_csv(raw_path, index=False)

print("Saved:", raw_path)
print("Rows:", len(raw_df))

Saved: /content/drive/MyDrive/finance-rag-analyst/data/raw/sec_filings_raw.csv
Rows: 30


In [35]:
# Raw filing quality checks

raw_df["char_count"] = raw_df["text"].str.len()
raw_df["word_count"] = raw_df["text"].str.split().str.len()

print("Raw filings:", len(raw_df))
print("Companies:", raw_df["ticker"].nunique())
print("Forms:", raw_df["form_type"].value_counts().to_dict())
print("Median chars:", int(raw_df["char_count"].median()))
print("Min chars:", int(raw_df["char_count"].min()))
print("Max chars:", int(raw_df["char_count"].max()))

display(
    raw_df[[
        "ticker", "company", "form_type", "filing_date",
        "report_date", "char_count", "word_count", "source_url"
    ]].sort_values(["ticker", "form_type", "filing_date"])
)

Raw filings: 30
Companies: 5
Forms: {'10-K': 15, '10-Q': 15}
Median chars: 203151
Min chars: 56972
Max chars: 354483


,ticker,company,form_type,filing_date,report_date,char_count,word_count,source_url
8,AAPL,Apple Inc.,10-K,2023-11-03,2023-09-30,200598,30650,https://www.sec.gov/Archives/edgar/data/320193...
7,AAPL,Apple Inc.,10-K,2024-11-01,2024-09-28,204321,31122,https://www.sec.gov/Archives/edgar/data/320193...
6,AAPL,Apple Inc.,10-K,2025-10-31,2025-09-27,206727,31482,https://www.sec.gov/Archives/edgar/data/320193...
11,AAPL,Apple Inc.,10-Q,2025-08-01,2025-06-28,59899,9689,https://www.sec.gov/Archives/edgar/data/320193...
10,AAPL,Apple Inc.,10-Q,2026-01-30,2025-12-27,56972,9033,https://www.sec.gov/Archives/edgar/data/320193...
9,AAPL,Apple Inc.,10-Q,2026-05-01,2026-03-28,84747,13347,https://www.sec.gov/Archives/edgar/data/320193...
26,AMZN,AMAZON COM INC,10-K,2024-02-02,2023-12-31,283111,42988,https://www.sec.gov/Archives/edgar/data/101872...
25,AMZN,AMAZON COM INC,10-K,2025-02-07,2024-12-31,280184,42501,https://www.sec.gov/Archives/edgar/data/101872...
24,AMZN,AMAZON COM INC,10-K,2026-02-06,2025-12-31,283477,43012,https://www.sec.gov/Archives/edgar/data/101872...
29,AMZN,AMAZON COM INC,10-Q,2025-08-01,2025-06-30,172099,26398,https://www.sec.gov/Archives/edgar/data/101872...


Flag sus filings

In [36]:
# Flag suspicious filings

suspicious_df = raw_df[
    (raw_df["char_count"] < 50_000) |
    (raw_df["word_count"] < 5_000) |
    (raw_df["text"].isna())
].copy()

print("Suspicious filings:", len(suspicious_df))

display(
    suspicious_df[[
        "ticker", "form_type", "filing_date",
        "char_count", "word_count", "source_url"
    ]]
)

Suspicious filings: 0


,ticker,form_type,filing_date,char_count,word_count,source_url


Save final raw dataset

In [37]:
# Save final Notebook 1 output

raw_final_path = f"{PROCESSED_DIR}/sec_filings_raw_final.csv"

raw_df.to_csv(raw_final_path, index=False)

print("Saved final raw dataset:")
print(raw_final_path)
print("Rows:", len(raw_df))

Saved final raw dataset:
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_raw_final.csv
Rows: 30


Final Notebook 1 validation

In [38]:
# Final validation

required_cols = [
    "ticker",
    "company",
    "cik",
    "form_type",
    "filing_date",
    "report_date",
    "accession_number",
    "primary_document",
    "source_url",
    "text",
    "char_count",
    "word_count"
]

missing = [col for col in required_cols if col not in raw_df.columns]

assert not missing, f"Missing columns: {missing}"
assert len(raw_df) == 30, f"Expected 30 filings, got {len(raw_df)}"
assert raw_df["ticker"].nunique() == 5, "Expected 5 companies"
assert raw_df["char_count"].median() > 100_000, "Median filing text looks too short"
assert raw_df["source_url"].notna().all(), "Some source URLs are missing"
assert raw_df["text"].notna().all(), "Some filing texts are missing"

print("Notebook 1 completed successfully.")
print("Use this file for Notebook 2:")
print(raw_final_path)

Notebook 1 completed successfully.
Use this file for Notebook 2:
/content/drive/MyDrive/finance-rag-analyst/data/processed/sec_filings_raw_final.csv
